## 1. Environment Setup

In [ ]:
# Core libraries
import os
import json
import time
import random
import shutil
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.restoration import wiener, richardson_lucy

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
COCO_ROOT = PROJECT_ROOT / "coco"

BLUR_DIR = COCO_ROOT / "blur"
SHARP_DIR = COCO_ROOT / "sharp"
LABEL_JSON_PATH = COCO_ROOT / "label" / "instances_val2017.json"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
DEBLUR_ROOT = OUTPUT_ROOT / "deblur"
ANALYSIS_ROOT = OUTPUT_ROOT / "analysis"
PROCESSED_ROOT = OUTPUT_ROOT / "processed_dataset"
RUNS_ROOT = OUTPUT_ROOT / "runs"

for p in [OUTPUT_ROOT, DEBLUR_ROOT, ANALYSIS_ROOT, PROCESSED_ROOT, RUNS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BLUR_DIR exists:", BLUR_DIR.exists())
print("SHARP_DIR exists:", SHARP_DIR.exists())
print("LABEL_JSON_PATH exists:", LABEL_JSON_PATH.exists())

## 2. Load Blur and Sharp Images

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

blur_files = sorted([p.name for p in BLUR_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])
sharp_files = sorted([p.name for p in SHARP_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])

blur_set = set(blur_files)
sharp_set = set(sharp_files)

common_files = sorted(blur_set & sharp_set)
blur_only = sorted(blur_set - sharp_set)
sharp_only = sorted(sharp_set - blur_set)

pair_df = pd.DataFrame({
    "file_name": common_files
})

print("Blur images  :", len(blur_files))
print("Sharp images :", len(sharp_files))
print("Matched pairs:", len(common_files))
print("Blur only    :", len(blur_only))
print("Sharp only   :", len(sharp_only))

if blur_only:
    print("\nFirst 10 blur-only files:", blur_only[:10])

if sharp_only:
    print("\nFirst 10 sharp-only files:", sharp_only[:10])

pair_df.head()

## 3. Pair Consistency Check

In [ ]:
def read_bgr(path: Path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Failed to read image: {path}")
    return img

size_rows = []
bad_pairs = []

for file_name in common_files:
    blur_path = BLUR_DIR / file_name
    sharp_path = SHARP_DIR / file_name
    try:
        blur_img = read_bgr(blur_path)
        sharp_img = read_bgr(sharp_path)
        size_rows.append({
            "file_name": file_name,
            "blur_h": blur_img.shape[0],
            "blur_w": blur_img.shape[1],
            "sharp_h": sharp_img.shape[0],
            "sharp_w": sharp_img.shape[1],
            "same_shape": blur_img.shape[:2] == sharp_img.shape[:2]
        })
    except Exception as e:
        bad_pairs.append((file_name, str(e)))

size_df = pd.DataFrame(size_rows)
display(size_df.head())

print("Pairs with same shape:", int(size_df["same_shape"].sum()), "/", len(size_df))
print("Unreadable pairs:", len(bad_pairs))
if bad_pairs:
    print(bad_pairs[:5])